# Brute Force Approach for Minimum fuel Trajectories in Earth Moon System

This notebook applies a brute force approach to solve the problem of launching a rocket from Low Earth Orbit (LEO) to Low Moon Orbit (LMO).

Two impulsive burns are appied. One at LEO and one at LMO.

The time of flight and phase of departure are also optimized

### Imports

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from cr3bp import (
    create_earth_moon_system,
    grid_search_method,
    shrink_ranges
)

### Initialize the System / Problem

In [2]:
# Create the Earth-Moon system using the cr3bp module
em = create_earth_moon_system()
print(em.info())

CR3BP System Information:
  Primary 1 mass: 5.972e+24 kg
  Primary 2 mass: 7.342e+22 kg
  Primary 1 radius: 6.371e+06 m
  Primary 2 radius: 1.737e+06 m
  Total mass: 6.045e+24 kg
  Distance: 3.844e+08 m (384400.0 km)
  Mass parameter μ: 0.012145

Characteristic scales:
  Length (l*): 3.844e+08 m (384400.0 km)
  Time (t*): 3.752e+05 s (4.343 days)
  Velocity (v*): 1.025e+03 m/s (1.025 km/s)
  Acceleration (a*): 2.731e-03 m/s^2
  Period: 27.285 days
None


In [3]:
# Define the LEO and LMO altitudes in meters
leo_alt_m=463e3
lmo_alt_m=100e3

In [4]:
# 10km in natural units
print(f"10km in natural units = {10e3/em.l_star}")

10km in natural units = 2.6014568158168575e-05


## Run the Optimization Method

In [ ]:
grid_size = (20, 20, 25, 30)  # (num_theta, num_delta_v, num_delta_v_angle, num_tof)

### Iteration 1 - Coarse Grid Search

In [5]:
dec_var_ranges = [[3.9, 4.0], [3.0, 3.1], [0.05, 0.12], [0.65, 0.75]]

In [ ]:
optimals_iteration1 = Path("optimals_iteration1.npy")

if optimals_iteration1.exists():
    optimals_iteration1 = np.load(optimals_iteration1, allow_pickle=True)
    print("Loaded existing results")
    print(optimals_iteration1)
else:
    results_df = grid_search_method(em, grid_size, dec_var_ranges, 2.6e-5, leo_alt_m, lmo_alt_m)
    optimals_iteration1 = results_df.iloc[0][['theta', 'delta_v', 'delta_v_angle', 'tof']].values
    np.save("optimals_iteration1.npy", optimals_iteration1)
    np.save("optimals_iteration1_df.npy", results_df)   
    print("Ran grid search and saved")
    print(optimals_iteration1)
    print(results_df)

Loaded existing results
[3.98947368 3.02105263 0.05875    0.74310345]


### Iteration 2

optimals_iteration2_df = np.load("optimals_iteration2_df.npy", allow_pickle=True)
df = pd.DataFrame(optimals_iteration2_df)
df.columns = ['theta', 'delta_v', 'delta_v_angle', 'tof', 'Objective','Constraint']
print(df)
optimals_iteration2 = df.iloc[2][['theta', 'delta_v', 'delta_v_angle', 'tof']].values
np.save("optimals_iteration2.npy", optimals_iteration2)
print(optimals_iteration2)

In [ ]:
optimals_iteration2 = Path("optimals_iteration2.npy")

if optimals_iteration2.exists():
    dec_var_ranges2 = shrink_ranges(optimals_iteration1, dec_var_ranges, shrink_factor=0.5)
    print(f"New dec_var_ranges2: {dec_var_ranges2}")
    optimals_iteration2 = np.load(optimals_iteration2, allow_pickle=True)
    print("Loaded existing results")
    print(optimals_iteration2)
else:
    dec_var_ranges2 = shrink_ranges(optimals_iteration1, dec_var_ranges, shrink_factor=0.5)
    print(f"New dec_var_ranges2: {dec_var_ranges2}")
    results_df2 = grid_search_method(em, grid_size, dec_var_ranges2, 2.6e-5, leo_alt_m, lmo_alt_m)
    optimals_iteration2 = results_df2.iloc[0][['theta', 'delta_v', 'delta_v_angle', 'tof']].values
    np.save("optimals_iteration2.npy", optimals_iteration2)
    np.save("optimals_iteration2_df.npy", results_df2)   
    print("Ran grid search and saved")
    print(optimals_iteration2)
    print(results_df2)

New dec_var_ranges2: [[np.float64(3.9644736842105264), 4.0], [3.0, np.float64(3.0460526315789473)], [0.05, np.float64(0.07625)], [np.float64(0.718103448275862), 0.75]]
Loaded existing results
[3.97382271 3.02423823 0.0696875  0.72910226]


### Iteration 3

In [8]:
optimals_iteration3 = Path("optimals_iteration3.npy")

if optimals_iteration3.exists():
    optimals_iteration3 = np.load(optimals_iteration3, allow_pickle=True)
    print("Loaded existing results")
    print(optimals_iteration3)
else:
    dec_var_ranges3 = shrink_ranges(optimals_iteration2, dec_var_ranges2, shrink_factor=0.5)
    print(f"New dec_var_ranges3: {dec_var_ranges3}")
    print(f"Optimal iteration 2: {optimals_iteration2}")
    results_df3 = grid_search_method(em, grid_size, dec_var_ranges3, 3e-6, leo_alt_m, lmo_alt_m)
    optimals_iteration3 = results_df3.iloc[0][['theta', 'delta_v', 'delta_v_angle', 'tof']].values
    np.save("optimals_iteration3.npy", optimals_iteration3)
    np.save("optimals_iteration3_df.npy", results_df3)   
    print("Ran grid search and saved")
    print(optimals_iteration3)
    print(results_df3)

New dec_var_ranges3: [[np.float64(3.964941135734072), np.float64(3.982704293628809)], [np.float64(3.0127250692520775), np.float64(3.035751385041551)], [np.float64(0.063125), np.float64(0.07625)], [np.float64(0.7211281212841855), np.float64(0.7370763971462545)]]
Optimal iteration 2: [3.97382271 3.02423823 0.0696875  0.72910226]
Performing grid search with 20 x 20 x 25 x 30 = 300000 grid points...
Iteration: 21592/300000
Grid point satisfies constraint: Theta=3.97 rad, Delta_v=3.02, Delta_v_angle=0.07 rad, TOF=0.73 s => Distance to LMO=-0.15 km, Total Delta_v=6.30 km/s
New optimal found: Delta_v=3.10 km/s, Theta=3.97 rad, Delta_v_angle=0.07 rad, TOF=0.73 s, Distance to LMO=-0.15 km
Iteration: 23073/300000
Grid point satisfies constraint: Theta=3.97 rad, Delta_v=3.02, Delta_v_angle=0.07 rad, TOF=0.72 s => Distance to LMO=0.49 km, Total Delta_v=4.04 km/s
New optimal found: Delta_v=3.10 km/s, Theta=3.97 rad, Delta_v_angle=0.07 rad, TOF=0.72 s, Distance to LMO=0.49 km
Iteration: 51626/300000

In [ ]:
results_df3['distance_to_lmo'] = results_df3['distance_to_lmo'] * em.l_star * 1e-3